# 01 - Data Exploration

This notebook explores the project datasets before formal analysis.

Goals:
- Inspect available tables and columns
- Check quality and missingness
- Review country-specific patterns
- Prepare assumptions for downstream analysis

In [1]:
from pathlib import Path
import sqlite3

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

In [ ]:
BASE_DIR = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
DB_PATH = BASE_DIR / 'database' / 'database.db'

assert DB_PATH.exists(), f'Database not found: {DB_PATH}'

with sqlite3.connect(DB_PATH) as conn:
    tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name", conn)

tables

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    norway_raw = pd.read_sql_query('SELECT * FROM norway LIMIT 5', conn)
    usa_raw = pd.read_sql_query('SELECT * FROM usa LIMIT 5', conn)
    ph_raw = pd.read_sql_query('SELECT * FROM philippines LIMIT 5', conn)

display(norway_raw.head())
display(usa_raw.head())
display(ph_raw.head())

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    norway_clean = pd.read_sql_query('SELECT * FROM norway_inequality_indicators_clean ORDER BY year', conn)
    usa_clean = pd.read_sql_query('SELECT * FROM usa_clean', conn)
    ph_clean = pd.read_sql_query('SELECT * FROM philippines_clean', conn)

quality = pd.DataFrame({
    'table': ['norway_inequality_indicators_clean', 'usa_clean', 'philippines_clean'],
    'rows': [len(norway_clean), len(usa_clean), len(ph_clean)],
    'cols': [norway_clean.shape[1], usa_clean.shape[1], ph_clean.shape[1]],
    'missing_values': [norway_clean.isna().sum().sum(), usa_clean.isna().sum().sum(), ph_clean.isna().sum().sum()]
})

quality

In [ ]:
norway_plot = norway_clean[['year', 'gini_all_population', 'p90_p10_all_population', 's80_s20_all_population']].copy()
norway_plot = norway_plot.dropna(subset=['year'])
norway_plot = norway_plot.set_index('year')

ax = norway_plot.plot(figsize=(10, 5), marker='o', title='Norway inequality indicators over time')
ax.set_ylabel('Indicator value')
ax.grid(alpha=0.25)
plt.show()

In [ ]:
usa_focus = usa_clean[(usa_clean['income_type'] == 'MONEY INCOME') & (usa_clean['measure'].isin([
    'Gini index of income inequality',
    '90th/10th percentile income ratio',
    'Lowest quintile'
]))][['measure', 'year_2023_estimate', 'year_2024_estimate']]

usa_focus

In [ ]:
ph_2023 = ph_clean[['region', 'gini_2023']].dropna().sort_values('gini_2023', ascending=False)

top5 = ph_2023.head(5).rename(columns={'gini_2023': 'highest_gini_2023'})
bottom5 = ph_2023.tail(5).rename(columns={'gini_2023': 'lowest_gini_2023'})

display(top5)
display(bottom5)

## Exploration Notes

- Norway has the longest annual inequality series.
- USA provides a compact set of 2023/2024 comparison estimates.
- Philippines contains rich regional variation and a national series.

Next notebook focus: build comparable indicators and publication-ready visualizations.